# Lesson 8 — Grover's Search Algorithm

**Quantum Computing with Qiskit**

This notebook accompanies Lesson 8. Work through the cells in order. Every cell is meant to be run; modify the code freely and re-run to experiment.

**Topics covered:**
- Unstructured search and the phase oracle
- Geometric picture: 2D rotation toward the target
- Phase oracle and diffuser construction
- Amplitude amplification and optimal iterations
- Scaling analysis: quadratic speedup over classical search

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc --quiet

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Qiskit:             ", qiskit.__version__)
print("Qiskit Aer:         ", qiskit_aer.__version__)
print("Qiskit IBM Runtime: ", qiskit_ibm_runtime.__version__)

## 1. The Search Problem

**Problem:** Given a black-box $f: \{0,1\}^n \to \{0,1\}$ that returns 1 for exactly one input $\omega$ and 0 for all others, find $\omega$.

Classically: $O(N)$ queries on average, where $N = 2^n$.
Quantum (Grover): $O(\sqrt{N})$ queries.

The quantum phase oracle encodes $f$ as a phase flip:

$$U_\omega|x\rangle = (-1)^{f(x)}|x\rangle$$

It flips the sign of the target state $|\omega\rangle$ and leaves all other basis states unchanged. Applied to a uniform superposition, it evaluates $f$ on all $N$ inputs simultaneously. The phase difference is invisible to measurement — the job of the diffuser step is to convert it into an amplitude difference.

The overall strategy is **amplitude amplification**: repeatedly apply oracle + diffuser to boost the target's probability amplitude while suppressing all others.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
import numpy as np

sim = AerSimulator()

# Demonstrate that phases are invisible before interference
n = 2
target = '11'

# Uniform superposition
qc_demo = QuantumCircuit(n)
qc_demo.h(range(n))
sv_before = Statevector.from_label('0' * n).evolve(qc_demo)

# Apply phase flip on |11>
qc_demo.cz(0, 1)   # CZ flips sign of |11> only
sv_after = Statevector.from_label('0' * n).evolve(qc_demo)

print(f"n={n}, target='{target}'\n")
print(f"{'state':>6}  {'amp before':>12}  {'amp after':>12}  {'prob before':>12}  {'prob after':>12}")
print("-" * 62)
for i in range(2**n):
    bitstr = format(i, f'0{n}b')   # qubit n-1 on left, qubit 0 on right (matches get_counts)
    ab = sv_before.data[i].real
    aa = sv_after.data[i].real
    print(f"  |{bitstr}>  {ab:+12.4f}  {aa:+12.4f}  {ab**2:12.4f}  {aa**2:12.4f}")
print()
print("Phases are invisible: probabilities are identical before and after the oracle.")

The oracle changes the amplitude of $|\omega\rangle$ from $+1/\sqrt{N}$ to $-1/\sqrt{N}$, but this sign flip does not change the squared magnitude. The diffuser step is what converts this phase difference into a measurable probability difference.

## 2. A Geometric Picture

The algorithm operates in a 2D subspace spanned by $|\omega\rangle$ (the target) and $|s'\rangle$ (the uniform superposition over all non-target states):

$$|s'\rangle = \frac{1}{\sqrt{N-1}}\sum_{x \ne \omega}|x\rangle$$

The initial state $|s\rangle = H^{\otimes n}|0\rangle^{\otimes n}$ lies in this plane:

$$|s\rangle = \sin(\theta)|\omega\rangle + \cos(\theta)|s'\rangle, \qquad \sin(\theta) = \frac{1}{\sqrt{N}}$$

Each Grover iteration is two reflections:
- **Oracle $U_\omega$**: reflects about $|s'\rangle$ (flips the $|\omega\rangle$ component)
- **Diffuser $D = 2|s\rangle\langle s| - I$**: reflects about $|s\rangle$

Two reflections compose to a rotation. Each iteration rotates by $2\theta$ toward $|\omega\rangle$.

After $k$ iterations:
$$P(\omega, k) = \sin^2\!\bigl((2k+1)\theta\bigr)$$

Optimal iterations: $k_{\text{opt}} \approx \frac{\pi}{4}\sqrt{N}$

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_grover_geometry(n, ax=None):
    """Visualise the 2D rotation picture for n qubits."""
    N = 2**n
    theta = np.arcsin(1 / np.sqrt(N))
    k_opt = round(np.pi / (4 * theta) - 0.5)

    angles = [theta + 2 * k * theta for k in range(k_opt + 2)]

    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))

    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.1, 1.1)
    ax.set_aspect('equal')
    ax.set_xlabel("|s'⟩ axis")
    ax.set_ylabel("|ω⟩ axis")
    ax.set_title(f"n={n}, N={N}, k_opt={k_opt}")

    # Draw unit arc
    arc_angles = np.linspace(0, np.pi / 2, 200)
    ax.plot(np.cos(arc_angles), np.sin(arc_angles), 'k--', alpha=0.3, lw=1)

    # Draw target axis
    ax.axvline(0, color='gray', lw=0.5, alpha=0.4)
    ax.axhline(0, color='gray', lw=0.5, alpha=0.4)

    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(angles)))
    for k, (angle, color) in enumerate(zip(angles, colors)):
        x = np.cos(angle)
        y = np.sin(angle)
        ax.annotate('', xy=(x, y), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
        label = f'k={k}' if k > 0 else 'k=0 (init)'
        ax.text(x * 1.05, y * 1.05, label, fontsize=7, color=color,
                ha='center', va='center')

    ax.text(0.02, 0.98, f'θ = {np.degrees(theta):.1f}°', transform=ax.transAxes,
            fontsize=9, va='top')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, n in zip(axes, [2, 3, 4]):
    plot_grover_geometry(n, ax)

plt.suptitle('Grover rotation in the 2D subspace', fontsize=12)
plt.tight_layout()
plt.show()

print("Each arrow is the state after k Grover iterations.")
print("The state rotates by 2θ per step toward the |ω⟩ axis.")

## 3. The Phase Oracle

The phase oracle must flip the sign of $|\omega\rangle$ and leave all other states unchanged. The construction has three steps:

1. Apply $X$ to each qubit where the target bit is `0`, mapping $|\omega\rangle \to |1\cdots 1\rangle$.
2. Apply a multi-controlled Z gate (MCZ) that flips the phase of $|1\cdots 1\rangle$. Built as $H \cdot \text{MCX} \cdot H$ on the last qubit.
3. Apply the same $X$ gates to undo step 1.

The bitstring convention follows Qiskit: `target[0]` is the MSB (highest qubit index), so `reversed(target)` maps `target[n-1-i]` to qubit $i$.

In [ ]:
def phase_oracle(n, target):
    """
    Phase oracle: U_omega|x> = (-1)^f(x)|x>.
    Flips the phase of |target> and leaves all other states unchanged.
    target: binary string in Qiskit bitstring order
            (target[0] = MSB = highest qubit index).
    """
    oracle = QuantumCircuit(n, name='Oracle')
    # Step 1: flip qubits where the target bit is '0'
    for i, bit in enumerate(reversed(target)):
        if bit == '0':
            oracle.x(i)
    # Step 2: multi-controlled Z via H · MCX · H on last qubit
    oracle.h(n - 1)
    oracle.mcx(list(range(n - 1)), n - 1)
    oracle.h(n - 1)
    # Step 3: undo the X flips
    for i, bit in enumerate(reversed(target)):
        if bit == '0':
            oracle.x(i)
    return oracle

# Draw the oracle for n=3, target='101'
n = 3
target = '101'
oracle = phase_oracle(n, target)
oracle.draw('mpl')

In [ ]:
# Verify: apply oracle to uniform superposition and inspect amplitudes
qc_check = QuantumCircuit(n)
qc_check.h(range(n))
qc_check.compose(oracle, inplace=True)

sv = Statevector.from_label('0' * n).evolve(qc_check)

print(f"Amplitudes after oracle (target = '{target}'):\n")
for i in range(2**n):
    bitstr = format(i, f'0{n}b')   # qubit n-1 on left, qubit 0 on right (matches get_counts)
    amp = sv.data[i].real
    marker = '  <-- phase flipped' if bitstr == target else ''
    print(f"  |{bitstr}>: {amp:+.4f}{marker}")

print()
print("All probabilities remain 1/8 — the phase flip is invisible until interference.")

## 4. The Diffuser

The diffuser $D = 2|s\rangle\langle s| - I$ reflects the state about the initial uniform superposition $|s\rangle$. It decomposes as:

$$D = H^{\otimes n}\,(2|0\rangle\langle 0| - I)\,H^{\otimes n}$$

The inner operator flips the sign of every state except $|0\cdots 0\rangle$, which is equivalent (up to global phase) to a phase flip of $|1\cdots 1\rangle$ after applying $X$ to all qubits.

**Inversion about the mean:** if the amplitudes before the diffuser are $a_0, \ldots, a_{N-1}$ with mean $\mu$, the diffuser maps $a_x \mapsto 2\mu - a_x$. After the oracle, the target amplitude is well below average (it has been negated). The diffuser reflects it to well above average.

In [ ]:
def diffuser(n):
    """
    Grover diffuser: D = 2|s><s| - I  where |s> = H^n|0>.
    Implements inversion about the mean.
    """
    diff = QuantumCircuit(n, name='Diffuser')
    diff.h(range(n))
    diff.x(range(n))
    # Phase flip of |1...1> via H · MCX · H on the last qubit
    diff.h(n - 1)
    diff.mcx(list(range(n - 1)), n - 1)
    diff.h(n - 1)
    diff.x(range(n))
    diff.h(range(n))
    return diff

diff = diffuser(3)
diff.draw('mpl')

In [ ]:
# Demonstrate inversion about the mean for n=2
# Start with amplitudes that have one negated entry and show how diffuser redistributes
n_demo = 2
target_demo = '01'

# Uniform superposition
qc_d = QuantumCircuit(n_demo)
qc_d.h(range(n_demo))
sv0 = Statevector.from_label('0' * n_demo).evolve(qc_d)

# After oracle
qc_d.compose(phase_oracle(n_demo, target_demo), inplace=True)
sv1 = Statevector.from_label('0' * n_demo).evolve(qc_d)

# After oracle + diffuser
qc_d.compose(diffuser(n_demo), inplace=True)
sv2 = Statevector.from_label('0' * n_demo).evolve(qc_d)

print(f"n={n_demo}, target='{target_demo}'  (1 Grover iteration)\n")
print(f"{'state':>6}  {'initial':>10}  {'after oracle':>13}  {'after diffuser':>15}")
print("-" * 52)
for i in range(2**n_demo):
    bitstr = format(i, f'0{n_demo}b')   # qubit n-1 on left, qubit 0 on right (matches get_counts)
    a0 = sv0.data[i].real
    a1 = sv1.data[i].real
    a2 = sv2.data[i].real
    marker = ' <--' if bitstr == target_demo else ''
    print(f"  |{bitstr}>  {a0:+10.4f}  {a1:+13.4f}  {a2:+15.4f}{marker}")

print()
mu1 = sv1.data.real.mean()
print(f"Mean amplitude after oracle: {mu1:.4f}")
print("Diffuser maps each a_x -> 2μ - a_x. Target was below average, now above.")

## 5. Running Grover's Algorithm

The full Grover circuit applies the uniform superposition once, then repeats oracle + diffuser for the optimal number of iterations.

In [ ]:
def grover_circuit(n, target, num_iterations):
    """
    Build Grover's circuit (no measurement).
    n: number of qubits.
    target: binary string in Qiskit bitstring order.
    num_iterations: number of oracle + diffuser cycles.
    """
    qc = QuantumCircuit(n)
    qc.h(range(n))   # uniform superposition
    qc.barrier()

    oracle = phase_oracle(n, target)
    diff = diffuser(n)

    for _ in range(num_iterations):
        qc.compose(oracle, inplace=True)
        qc.compose(diff, inplace=True)
        qc.barrier()
    return qc


def optimal_iterations(n):
    """Optimal number of Grover iterations for N=2^n and one marked item."""
    N = 2**n
    theta = np.arcsin(1 / np.sqrt(N))
    return round(np.pi / (4 * theta) - 0.5)


# Run for n=3, target='101'
n = 3
target = '101'
k = optimal_iterations(n)
print(f"n={n}, N={2**n}, optimal iterations k={k}\n")

qc = grover_circuit(n, target, k)

# Statevector simulation
sv = Statevector(qc)
probs = sv.probabilities_dict()

print(f"Statevector probabilities after {k} iteration(s):\n")
for bitstr, p in sorted(probs.items(), key=lambda x: -x[1]):
    marker = '  <-- target' if bitstr == target else ''
    print(f"  |{bitstr}>: {p:.4f}{marker}")

In [ ]:
# Run with shots to simulate a measurement
qc_m = qc.copy()
qc_m.measure_all()

counts = sim.run(qc_m, shots=4096).result().get_counts()
top = max(counts, key=counts.get)
print(f"Most frequent outcome: '{top}' ({counts[top]} / 4096 shots = {counts[top]/4096:.3f})")
print(f"Target was: '{target}'  Match: {top == target}")

In [ ]:
# Visualise the measurement histogram
from qiskit.visualization import plot_histogram

plot_histogram(counts, title=f"Grover n={n}, target='{target}', k={k}")

## 6. Optimal Iterations and Scaling

The success probability oscillates sinusoidally: $P(\omega, k) = \sin^2((2k+1)\theta)$. Running past the optimal $k$ decreases success probability.

In [ ]:
# Amplitude evolution plot
n = 3
target = '101'
N = 2**n
theta = np.arcsin(1 / np.sqrt(N))
max_iters = 10

# Simulated probabilities at each k
sim_probs = []
for k in range(max_iters + 1):
    qc_k = grover_circuit(n, target, k)
    p = Statevector(qc_k).probabilities_dict().get(target, 0)
    sim_probs.append(p)

# Theoretical curve
k_cont = np.linspace(0, max_iters, 300)
theory = np.sin((2 * k_cont + 1) * theta) ** 2

k_opt = optimal_iterations(n)

plt.figure(figsize=(8, 4))
plt.plot(k_cont, theory, label=r'Theory: $\sin^2((2k+1)\theta)$')
plt.scatter(range(max_iters + 1), sim_probs, zorder=5, label='Simulated')
plt.axvline(k_opt, color='red', linestyle=':', label=f'Optimal k={k_opt}')
plt.xlabel('Grover iterations k')
plt.ylabel(f"P('{target}')")
plt.title(f"Amplitude evolution: n={n}, N={N}, target='{target}'")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Quadratic speedup: compare quantum vs classical query counts
print(f"{'n':>4}  {'N':>8}  {'Quantum k':>10}  {'Classical N/2':>14}")
print(f"{'':->4}  {'':->8}  {'':->10}  {'':->14}")

for n in [2, 3, 4, 6, 8, 10, 12]:
    N = 2**n
    k = optimal_iterations(n)
    print(f"{n:>4}  {N:>8}  {k:>10}  {N//2:>14}")

In [ ]:
# End-to-end verification across problem sizes
print("Grover's algorithm: end-to-end verification across n\n")
for n in range(2, 7):
    N = 2**n
    target = format(np.random.randint(N), f'0{n}b')
    k = optimal_iterations(n)

    qc = grover_circuit(n, target, k)
    qc_m = qc.copy()
    qc_m.measure_all()

    counts = sim.run(qc_m, shots=2048).result().get_counts()
    found = max(counts, key=counts.get)
    success = found == target
    p_top = counts[found] / 2048
    print(f"  n={n}, N={N:>4}, k={k}, target='{target}', "
          f"found='{found}', P={p_top:.3f}, correct={success}")

## 7. Exercises

### Exercise 1: Exact solution for n=2

For $n=2$, $N=4$, $\theta = \arcsin(1/2) = \pi/6$. One iteration gives $(2 \cdot 1 + 1)\theta = \pi/2$, so $P = \sin^2(\pi/2) = 1$ exactly.

1. Pick any target from `'00'`, `'01'`, `'10'`, `'11'`.
2. Build the Grover circuit with exactly 1 iteration and measure all.
3. Verify that the target is found with probability 1 (4096 shots, all the same outcome).
4. What happens if you run 2 iterations instead of 1? Compute the expected probability and verify it experimentally.

In [ ]:
# Exercise 1: your solution here
n = 2
target = '10'   # change this to any 2-bit string

# TODO: run with k=1, verify P=1
# TODO: run with k=2, compute expected P and compare

### Exercise 2: Oracle with multiple targets

Grover's algorithm also works when there are $M > 1$ marked items. The angle becomes $\theta = \arcsin(\sqrt{M/N})$ and the optimal iterations become $k \approx \frac{\pi}{4}\sqrt{N/M}$.

1. Build a phase oracle for $n=3$ that marks **two** targets: `'011'` and `'110'`. The oracle should flip the phase of both states.
   Hint: compose two single-target oracles, or use a controlled-phase approach.
2. Compute the expected $\theta$ and $k_{\text{opt}}$ for $M=2$, $N=8$.
3. Run the algorithm and measure. Both targets should appear with roughly equal (and high) probability.
4. What is the expected success probability at $k_{\text{opt}}$?

In [ ]:
# Exercise 2: your solution here
n = 3
targets = ['011', '110']   # two marked items

# TODO: build a two-target oracle
def two_target_oracle(n, targets):
    oracle = QuantumCircuit(n, name='Oracle')
    for t in targets:
        oracle.compose(phase_oracle(n, t), inplace=True)
    return oracle

# TODO: compute theta and k_opt for M=2
M = 2
N = 2**n
# theta = ...
# k_opt = ...

# TODO: build and run the circuit, show histogram

### Exercise 3: Visualising amplitude amplification step by step

For $n=3$ and `target='011'`:

1. Compute the statevector after 0, 1, 2, and 3 iterations.
2. For each iteration, extract the amplitude of the target state and all non-target states.
3. Plot the amplitudes as a bar chart for each step, with the target state highlighted.
4. At which iteration does the target amplitude peak? What happens at iteration 3?

This visualises the inversion-about-the-mean mechanism directly in the amplitude space.

In [ ]:
# Exercise 3: your solution here
n = 3
target = '011'
max_k = 3

fig, axes = plt.subplots(1, max_k + 1, figsize=(16, 4))

for k, ax in enumerate(axes):
    qc_k = grover_circuit(n, target, k)
    sv_k = Statevector(qc_k)

    # TODO: extract amplitudes
    # TODO: plot bar chart, highlight target state
    ax.set_title(f'k={k}')
    ax.set_xlabel('state')
    ax.set_ylabel('amplitude')

plt.suptitle(f"Amplitude amplification: n={n}, target='{target}'")
plt.tight_layout()
plt.show()

### Exercise 4: Grover with a noisy simulator

Run Grover's algorithm for $n=4$, `target='1010'` on a noisy simulator.

1. Use `AerSimulator` with a depolarizing noise model: `depolarizing_error(p, 1)` on single-qubit gates (`'h'`, `'x'`) and `depolarizing_error(p, 2)` on two-qubit gates (`'cx'`).
2. Run with $p \in \{0.001, 0.005, 0.01, 0.05\}$, 4096 shots each.
3. For each noise level, report the most frequently measured outcome and $P(\text{target})$.
4. At what noise level does the algorithm start failing? How does this compare to the depth of the circuit?

In [ ]:
# Exercise 4: your solution here
from qiskit_aer.noise import NoiseModel, depolarizing_error

n = 4
target = '1010'
k = optimal_iterations(n)
shots = 4096

def run_grover_noisy(n, target, k, p, shots=4096):
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p, 1), ['h', 'x'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p, 2), ['cx'])

    qc = grover_circuit(n, target, k)
    qc.measure_all()
    sim_noisy = AerSimulator(noise_model=noise_model)
    counts = sim_noisy.run(qc, shots=shots).result().get_counts()
    top = max(counts, key=counts.get)
    p_target = counts.get(target, 0) / shots
    return top, p_target

print(f"n={n}, target='{target}', k={k}\n")
print(f"  {'p':>8}  {'top answer':>12}  {'P(target)':>10}  {'correct':>8}")
print(f"  {'-'*8}  {'-'*12}  {'-'*10}  {'-'*8}")

for p in [0.001, 0.005, 0.01, 0.05]:
    top, p_t = run_grover_noisy(n, target, k, p, shots)
    print(f"  {p:>8.3f}  {top:>12s}  {p_t:>10.3f}  {str(top == target):>8}")

## Summary

In this lesson you:

- Verified that the phase oracle flips the sign of the target state while leaving all measurement probabilities unchanged: the phase difference is encoded but invisible until interference.
- Visualised the 2D rotation picture: each Grover iteration rotates the state vector by $2\theta$ toward $|\omega\rangle$ in the subspace spanned by $|\omega\rangle$ and $|s'\rangle$.
- Built the phase oracle using X gates, $H \cdot \text{MCX} \cdot H$ for multi-controlled Z, and demonstrated the inversion-about-the-mean effect of the diffuser.
- Ran the full Grover circuit for $n = 2$ through $n = 6$, confirming that the optimal iteration count $k \approx \frac{\pi}{4}\sqrt{N}$ gives success probability close to 1.
- Plotted the sinusoidal amplitude evolution and the quadratic speedup table showing $O(\sqrt{N})$ quantum queries against $O(N)$ classical queries.

**Lesson 9** puts these circuits on real quantum hardware. Transpilation maps abstract gates to the basis gate set of a specific device, inserts SWAP gates to route around connectivity constraints, and can significantly increase circuit depth. The lesson covers how to submit jobs, interpret noise-affected results, and apply readout error correction.